# ERK-KTR Full FOV Stimulation Pipeline (df_acquire API)

This notebook uses the **legacy `df_acquire` API** to build acquisition events via pandas DataFrames.
It is functionally equivalent to `Full_FOV_stim_ERKKTR_RTMSequence.ipynb`, which uses the newer **`RTMSequence` API** — a more concise, declarative way to define multi-phase experiments.

**When to use which API:**
- **RTMSequence** (recommended): phases are defined as `RTMSequence` objects and concatenated with `+`. Stim frames, ref frames, and channels are declared per-phase. See `Full_FOV_stim_ERKKTR_RTMSequence.ipynb`.
- **df_acquire** (this notebook): gives full control over the acquisition DataFrame. Useful for complex per-FOV condition assignments, wellplate layouts, or custom stim schedules that don't map cleanly to RTMSequence phases.

**Workflow overview:**
1. Initialize microscope and define channels / stimulation treatments
2. Configure the image processing pipeline (segmentation, tracking, feature extraction)
3. Select FOV positions in napari and build a `df_acquire` DataFrame per phase
4. Convert DataFrames to events and run the experiment
5. Post-process tracks into a single `exp_data.parquet`

### Import required libraries

**df_acquire API** uses `Channel`, `StimTreatment`, and `SegmentationMethod` from `data_structures`.

> **RTMSequence equivalent:** The RTMSequence API replaces `StimTreatment` with `PowerChannel` + `stim_channels`/`stim_frames` parameters on `RTMSequence`. See `Full_FOV_stim_ERKKTR_RTMSequence.ipynb`.

In [ ]:
import os
import time
from rtm_pymmcore.core.data_structures import (
    Channel,  # basic imaging channel (config, exposure, group)
    StimTreatment,  # df_acquire-specific: defines stim schedule per condition
    SegmentationMethod,  # shared with RTMSequence API
)
import rtm_pymmcore.core.utils as utils
from pprint import pprint
import pandas as pd

### Experimental Settings

**Microscope:** Jungfrau (no DMD). Change the import to use a different microscope.

> **RTMSequence equivalent:** Channel definitions are the same, but stimulation is defined via `PowerChannel` + `stim_channels`/`stim_frames` on `RTMSequence` instead of `StimTreatment` objects. The RTMSequence API also handles per-phase timing automatically — no need for manual `N_FRAMES_PHASE_*` or `TIME_PER_FOV` bookkeeping.

In [ ]:
from rtm_pymmcore.microscope.pertzlab.jungfrau import Jungfrau

mic = Jungfrau()
mic.mmc.setChannelGroup("TTL_ERK")

In [ ]:
## Configuration options - set experiment timing, storage and stimulation parameters

# General timing and frame counts:
N_FRAMES_PHASE_1 = 100  # number of timesteps in phase 1 (pre-drug)
N_FRAMES_PHASE_2 = 150  # number of timesteps in phase 2 (post-drug)
# RTMSequence equivalent: time_plan={"interval": 60.0, "loops": 100}

SLEEP_BEFORE_EXPERIMENT_START_in_H = (
    0  # delay before acquisition (hours); 0 = start immediately
)

# Timing for acquisition
TIME_BETWEEN_TIMESTEPS = 60  # seconds between timesteps
# RTMSequence equivalent: time_plan={"interval": 60.0, ...}
TIME_PER_FOV = 3.75  # seconds per FOV (used for scheduling estimates)
# RTMSequence equivalent: utils.apply_fov_batching(events, time_per_fov=3.75)

# Display / bookkeeping options (df_acquire-specific, not needed with RTMSequence):
ADD_STIM_EXPOSURE_GROUP = False  # set to True to save per-FOV last-stim-exposure info
REGULAR_SPACING_BETWEEN_STIMULATIONS = False  # True -> evenly spaced stim timings

## Storage path
base_path = "E:\\Alex"
experiment_name = "2025-11-13_Priming_FGFR_U0126"
path = os.path.join(base_path, experiment_name)

## Imaging channels -- acquired at every timepoint
# Channel(config, exposure, group) -- group defaults to None (uses mic.mmc channel group)
# RTMSequence equivalent: use PowerChannel(config, exposure, group, power) for light-source power control
channels = []
channels.append(Channel(config="miRFP", exposure=300))  # nuclear marker
channels.append(Channel(config="mScarlet3", exposure=200))  # ERK-KTR reporter

# Optocheck channel -- reference channel to verify optogenetic tool expression
# RTMSequence equivalent: ref_channels=(optocheck_channel,), ref_frames=[-1]
channel_optocheck = Channel(config="mCitrine", exposure=600)
optocheck_timepoints = (149,)  # absolute timestep indices within phase 2

# Experimental conditions assigned to FOVs
condition = ["Drug"]
n_fovs_per_well = None  # set to int for wellplate experiments

## Stimulation treatments (df_acquire-specific)
# StimTreatment defines the full stim schedule: which timesteps, what exposure, power, and channel.
# RTMSequence equivalent: stim_channels=(PowerChannel(...),), stim_frames=range(10, 100)
#   - stim_frames are per-phase (0-indexed), auto-offset on concatenation
#   - stim_exposure can be a single value or list (no need for auto_repeat_stim_exposure)

stim_phase_1 = [
    StimTreatment(
        treatment_name="Priming Phase 1 pre Drug",
        stim_timestep=(tuple(range(10, 100, 1))),  # stim at timesteps 10..99
        stim_exposure_list=(100,)
        * 90,  # per-timestep exposure (length must match stim_timestep)
        stim_power=10,
        stim_channel_name="CyanStim",
        stim_channel_group="TTL_ERK",
        stim_channel_device_name="Spectra",
        stim_channel_power_property_name="Cyan_Level",
    )
]

stim_phase_2 = [
    StimTreatment(
        treatment_name="Sustained Phase 2 post Drug",
        stim_timestep=(tuple(range(30, 120, 1))),
        stim_exposure_list=100,  # single value repeated when auto_repeat_stim_exposure=True
        auto_repeat_stim_exposure=True,
        stim_power=10,
        stim_channel_name="CyanStim",
        stim_channel_group="TTL_ERK",
        stim_channel_device_name="Spectra",
        stim_channel_power_property_name="Cyan_Level",
    )
]

# Print the stim schedule for confirmation
for stim_phase in [stim_phase_1, stim_phase_2]:
    utils.print_stim_exposures_timesteps(stim_phase)

## Pipeline components (shared between df_acquire and RTMSequence APIs)
from rtm_pymmcore.stimulation.base import StimWholeFOV
from rtm_pymmcore.tracking.trackpy import TrackerTrackpy
from rtm_pymmcore.feature_extraction.erk_ktr import FE_ErkKtr
from rtm_pymmcore.feature_extraction.optocheck import OptoCheckFE
from rtm_pymmcore.segmentation.remote import SegmentatorImagingServerKit

segmentators = [
    SegmentationMethod(
        name="labels",
        segmentation_class=SegmentatorImagingServerKit(
            server="http://izbniesen.izb.unibe.ch:8001",
            algorithm="StarDist (2D)",
            min_size=50,
            model_param={"model_name": "MCF10A_v01"},
        ),
        use_channel=0,
        save_tracked=True,
    )
]

stimulator = StimWholeFOV()
feature_extractor = FE_ErkKtr("labels")
tracker = TrackerTrackpy(search_range=50)
optocheck = OptoCheckFE(used_mask="labels")

from rtm_pymmcore.core.pipeline import ImageProcessingPipeline

# NOTE: parameter was renamed from feature_extractor_optocheck -> feature_extractor_ref
pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
    feature_extractor_ref=optocheck,  # renamed: was feature_extractor_optocheck in older versions
)

### GUI - Napari Micromanager

#### Load GUI

In [4]:
### Base GUI ###
from napari_micromanager import MainWindow
import napari

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
mm_wdg._mmc = mic.mmc
viewer.window.add_dock_widget(mm_wdg)
data_mda_fovs = None
load_from_file = False

### Build acquisition DataFrames

The df_acquire workflow builds one DataFrame per phase, then converts to events with `df_to_events()`.

**Key functions (df_acquire-specific):**
- `generate_fov_objects(mic, viewer=viewer)` -- reads FOV positions from napari MDA widget (alias for `generate_fov_positions`)
- `generate_df_acquire(fovs, ...)` -- builds a DataFrame with one row per (FOV, timestep), including channels, timing, and metadata
- `apply_stim_treatments_to_df_acquire(df, stim, ...)` -- maps `StimTreatment` objects onto the DataFrame rows

> **RTMSequence equivalent:** All of the above is replaced by declaring `RTMSequence(time_plan=..., stage_positions=..., channels=..., stim_channels=..., stim_frames=...)` and concatenating phases with `+`. See `Full_FOV_stim_ERKKTR_RTMSequence.ipynb`.

In [6]:
fovs = utils.generate_fov_objects(mic, viewer=viewer)


df_acquire = utils.generate_df_acquire(
    fovs,
    n_frames=N_FRAMES_PHASE_1,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    phase_name="PreDrug",
    phase_id=0,
    condition=condition,
)
df_acquire = utils.apply_stim_treatments_to_df_acquire(
    df_acquire,
    stim_phase_1,
    condition,
    n_fovs_per_well=n_fovs_per_well,
    add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
    regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
)
df_acquire

Total Experiment Time: 1.65h
Doing 3 experiment per stim condition


c:\Users\Jungfrau\Documents\alandolt\code\rtm-pymmcore\rtm_pymmcore\utils.py:177: FutureWarning: The `_dock_widgets` property is private and should not be used in any plugin code. Please use the `dock_widgets` property instead.
  data_mda_fovs = viewer.window._dock_widgets["MDA"].widget().value().stage_positions


,fov_object,fov,fov_x,fov_y,fov_name,timestep,time,channels,fname,cell_line,...,treatment_name,stim_timestep,stim_exposure_list,stim_power,stim_channel_name,stim_channel_group,stim_channel_device_name,stim_channel_power_property_name,stim_exposure,stim
0,<rtm_pymmcore.core.data_structures.Fov object at 0x...,0,-1.0,-0.7,0,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_00_00000,Drug,...,Priming Phase 1 pre Drug,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
1,<rtm_pymmcore.core.data_structures.Fov object at 0x...,1,-1.0,-0.7,1,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_00_00000,Drug,...,Priming Phase 1 pre Drug,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
2,<rtm_pymmcore.core.data_structures.Fov object at 0x...,2,-1.0,-0.7,2,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",002_00_00000,Drug,...,Priming Phase 1 pre Drug,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
3,<rtm_pymmcore.core.data_structures.Fov object at 0x...,0,-1.0,-0.7,0,1,60.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_00_00001,Drug,...,Priming Phase 1 pre Drug,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
4,<rtm_pymmcore.core.data_structures.Fov object at 0x...,1,-1.0,-0.7,1,1,60.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_00_00001,Drug,...,Priming Phase 1 pre Drug,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,<rtm_pymmcore.core.data_structures.Fov object at 0x...,1,-1.0,-0.7,1,98,5880.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_00_00098,Drug,...,Priming Phase 1 pre Drug,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,100.0,True
296,<rtm_pymmcore.core.data_structures.Fov object at 0x...,2,-1.0,-0.7,2,98,5880.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",002_00_00098,Drug,...,Priming Phase 1 pre Drug,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,100.0,True
297,<rtm_pymmcore.core.data_structures.Fov object at 0x...,0,-1.0,-0.7,0,99,5940.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_00_00099,Drug,...,Priming Phase 1 pre Drug,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,100.0,True
298,<rtm_pymmcore.core.data_structures.Fov object at 0x...,1,-1.0,-0.7,1,99,5940.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_00_00099,Drug,...,Priming Phase 1 pre Drug,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,100.0,True


In [ ]:
# generate_fov_objects is a backward-compat alias for generate_fov_positions
# RTMSequence equivalent: fov_positions = utils.generate_fov_positions(mic, viewer=viewer)
fovs = utils.generate_fov_objects(mic, viewer=viewer)

# Phase 1: build df_acquire with stim treatments applied
# RTMSequence equivalent: RTMSequence(time_plan={"interval": 60.0, "loops": 100}, ...)
df_acquire = utils.generate_df_acquire(
    fovs,
    n_frames=N_FRAMES_PHASE_1,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    phase_name="PreDrug",
    phase_id=0,
    condition=condition,
)
# Map StimTreatment objects onto DataFrame rows
# RTMSequence equivalent: stim_channels=(stim_channel,), stim_frames=range(10, 100)
df_acquire = utils.apply_stim_treatments_to_df_acquire(
    df_acquire,
    stim_phase_1,
    condition,
    n_fovs_per_well=n_fovs_per_well,
    add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
    regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
)
df_acquire

# Phase 2: separate df_acquire with optocheck channel on specific timepoints
# RTMSequence equivalent: RTMSequence(..., ref_channels=(optocheck_channel,), ref_frames=[-1])
df_acquire_2 = utils.generate_df_acquire(
    fovs,
    n_frames=N_FRAMES_PHASE_2,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    channel_optocheck=channel_optocheck,       # df_acquire passes optocheck as a parameter here
    optocheck_timepoints=optocheck_timepoints,  # absolute timestep indices (not per-phase like RTMSequence)
    phase_name="PostDrug",
    phase_id=1,
    condition=condition,
)
df_acquire_2 = utils.apply_stim_treatments_to_df_acquire(
    df_acquire_2,
    stim_phase_2,
    condition,
    n_fovs_per_well=n_fovs_per_well,
    add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
    regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
)
df_acquire_2

### Run experiment

The df_acquire API requires an extra step: convert each DataFrame to events with `df_to_events()`, then run each phase separately with its own `Controller`.

> **RTMSequence equivalent:** Events are generated directly by `phase_1 + phase_2` concatenation. A single `Controller` and `ctrl.run_experiment(events)` handles everything. No `df_to_events` conversion needed.

In [ ]:
from rtm_pymmcore.core.controller import Controller
from rtm_pymmcore.core.writers import OmeZarrWriter
from rtm_pymmcore.core.conversion import (
    df_to_events,
)  # df_acquire-specific: converts DataFrame -> list[RTMEvent]

# Optional: wait before starting
for _ in range(0, int(SLEEP_BEFORE_EXPERIMENT_START_in_H * 3600)):
    time.sleep(1)

# Disconnect napari live view so it does not interfere with the acquisition engine
try:
    mm_wdg._core_link.cleanup()
except:
    pass

# Phase 1: convert df_acquire -> events and run
events = df_to_events(df_acquire)
writer = OmeZarrWriter(storage_path=path)
ctrl = Controller(mic, pipeline, writer=writer)
ctrl.run_experiment(events, stim_mode="current")

In [ ]:
# Phase 2: convert second df_acquire -> events and run with a new Controller
# Note: df_acquire requires separate Controller instances per phase
# RTMSequence equivalent: phases are concatenated and run with a single ctrl.run_experiment()
events_2 = df_to_events(df_acquire_2)
ctrl_2 = Controller(mic, pipeline, writer=writer)
ctrl_2.run_experiment(events_2, stim_mode="current")

# Post-processing (same for both APIs)
mic.post_experiment()  # currently a no-op on Jungfrau, but good practice to call
time.sleep(10)

utils.generate_exp_data_from_tracks(path)  # merge per-FOV tracks into exp_data.parquet

# Reconnect napari live view
from napari_micromanager._core_link import CoreViewerLink

if "viewer" in locals():
    mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

In [18]:
### Manually reconnect pymmcore with napari-micromanager
from napari_micromanager._core_link import CoreViewerLink

mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

The following code block can be used to manually break the connection between GUI and Jupyter Notebook:


In [ ]:
### Break connection
# mm_wdg._core_link.cleanup()